In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# Calibration Stability Study

This notebook studies the stability of SABR calibration under three sources of modeling variation:

1. initial guess
2. fixed beta choice
3. shift choice in the shifted-SABR framework

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from swaption_pricing.calibration import (
    calibrate_sabr_across_beta_values,
    calibrate_sabr_for_multiple_initial_guesses,
    calibrate_shifted_sabr_across_shifts,
    calibration_diagnostics,
)
from swaption_pricing.sabr import SabrParams, sabr_implied_volatility, shifted_sabr_implied_volatility

## 1. Synthetic Positive-Rate Smile

In [ ]:
forward = 0.0400
expiry = 2.0
strikes = [0.0300, 0.0350, 0.0400, 0.0450, 0.0500]
true_params = SabrParams(alpha=0.0200, beta=0.50, rho=-0.25, nu=0.40)
market_vols = [sabr_implied_volatility(forward, strike, expiry, true_params) for strike in strikes]

## 2. Initial Guess Stability

In [ ]:
guess_results = calibrate_sabr_for_multiple_initial_guesses(
    forward,
    strikes,
    expiry,
    market_vols,
    beta=0.50,
    initial_guesses=[(0.0180, -0.10, 0.30), (0.0300, -0.50, 0.80), (0.0100, 0.20, 0.20)],
)

guess_df = pd.DataFrame([
    {
        'run': idx + 1,
        'alpha': result.params.alpha,
        'rho': result.params.rho,
        'nu': result.params.nu,
        'rmse': calibration_diagnostics(result).rmse,
    }
    for idx, result in enumerate(guess_results)
])
guess_df

## 3. Beta Stability

In [ ]:
beta_results = calibrate_sabr_across_beta_values(
    forward,
    strikes,
    expiry,
    market_vols,
    beta_values=[0.0, 0.5, 1.0],
    initial_guess=(0.0180, -0.10, 0.30),
)

beta_df = pd.DataFrame([
    {
        'beta': result.params.beta,
        'alpha': result.params.alpha,
        'rho': result.params.rho,
        'nu': result.params.nu,
        'rmse': calibration_diagnostics(result).rmse,
    }
    for result in beta_results
])
beta_df

In [ ]:
ax = beta_df.plot(x='beta', y='rmse', marker='o', figsize=(8, 4), title='Calibration RMSE vs Fixed Beta')
ax.set_ylabel('RMSE')
plt.show()

## 4. Shift Stability Under Negative Rates

In [ ]:
negative_forward = -0.0020
negative_strikes = [-0.0100, -0.0050, -0.0010, 0.0020, 0.0060]
base_shift = 0.03
shifted_market_vols = [
    shifted_sabr_implied_volatility(negative_forward, strike, expiry, true_params, base_shift)
    for strike in negative_strikes
]

shift_results = calibrate_shifted_sabr_across_shifts(
    negative_forward,
    negative_strikes,
    expiry,
    shifted_market_vols,
    beta=0.50,
    shifts=[0.02, 0.03, 0.05],
    initial_guess=(0.0180, -0.10, 0.30),
)

shift_df = pd.DataFrame([
    {
        'shift': shift,
        'alpha': result.params.alpha,
        'rho': result.params.rho,
        'nu': result.params.nu,
        'rmse': calibration_diagnostics(result).rmse,
    }
    for shift, result in shift_results
])
shift_df

In [ ]:
ax = shift_df.plot(x='shift', y='rmse', marker='o', figsize=(8, 4), title='Shifted-SABR Calibration RMSE vs Shift')
ax.set_ylabel('RMSE')
plt.show()

## 5. Interpretation

- Similar recovered parameters across initial guesses indicate numerical stability.
- RMSE across fixed beta values reveals how much fit quality depends on model convention.
- RMSE and parameter variation across shifts show that shift is a modeling choice that can move calibration outputs, not just a numerical workaround.

This makes calibration stability part of model risk, not just optimization hygiene.